# BLAST: Sequence Similarity Searching

**Tier 2 -- Core Bioinformatics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain how BLAST works (the seed-and-extend heuristic) and why it is needed
2. Choose the correct BLAST variant for your data (blastn, blastp, blastx, tblastn, tblastx)
3. Run BLAST searches via the NCBI web interface and programmatically with BioPython
4. Parse BLAST results in both XML and tabular formats
5. Interpret E-values, bit scores, percent identity, and query coverage
6. Use PSI-BLAST for detecting remote homologs
7. Apply best practices for database selection, filtering, and parameter tuning

---

## Why this notebook matters

BLAST (Basic Local Alignment Search Tool) is the most widely used bioinformatics tool. It runs billions of searches every year on NCBI's servers alone. Understanding how BLAST works — the seed-and-extend heuristic, word neighborhoods, the five program variants, E-value calculation, and PSI-BLAST for remote homologs — is essential for interpreting search results correctly and not being misled by high-identity but statistically weak hits, or missing genuinely homologous but divergent proteins.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section
5. If a code cell requires internet access (Entrez, PDB download), it is marked — these can be skipped if offline

## Complicated moments explained

- **Choosing the right BLAST program**: The five programs differ by query type and database type. blastn: nucleotide vs nucleotide. blastp: protein vs protein. blastx: nucleotide query (translated) vs protein database — use when you have a novel nucleotide sequence and want to find protein homologs. tblastn: protein query vs nucleotide database (translated) — use when you want to find unannotated protein-coding genes. tblastx: nucleotide query (translated) vs nucleotide database (translated) — slowest, rarely needed.
- **E-value depends on database size**: The same alignment gives a much lower (better) E-value against a small database than against nr. A hit with E=1e-5 in SwissProt is not the same as E=1e-5 in nr. Always report which database you searched.
- **E-value is not a p-value**: E-value is the *expected number of hits* with that score by chance. E=0.01 means you expect 0.01 random hits of that quality — not that there is a 1% chance of a false positive. For database searches, use E < 1e-5 as a typical significance threshold, but always consider alignment length and percent identity.
- **Low-complexity filtering (SEG/DUST)**: Sequences with low complexity (poly-A stretches, coiled-coil repeats, transmembrane helices) will generate thousands of spurious hits. Always leave low-complexity filtering enabled (the default) unless you have a specific reason to disable it.
- **PSI-BLAST convergence**: PSI-BLAST iterates until the set of hits stops changing. Watch for 'contamination' — if a false positive enters the PSSM in an early iteration, subsequent iterations become biased. Always verify hits biologically.

## Environment check (run this first)

In [ ]:
# Environment check
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Blast.Applications import NcbiblastpCommandline
from Bio.Align import substitution_matrices
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

blosum62 = substitution_matrices.load("BLOSUM62")
print("BioPython BLAST modules loaded.")

# Show the 5 BLAST programs at a glance
programs = {
    "blastn":  ("Nucleotide", "Nucleotide", "Standard DNA/RNA search"),
    "blastp":  ("Protein",    "Protein",    "Standard protein search"),
    "blastx":  ("Nucleotide (translated)", "Protein", "Annotate novel nucleotide seqs"),
    "tblastn": ("Protein",    "Nucleotide (translated)", "Find unannotated protein genes"),
    "tblastx": ("Nucleotide (translated)", "Nucleotide (translated)", "Cross-species DNA comparison"),
}
print(f"\n{'Program':<10} {'Query':<30} {'Database':<30} {'Use case'}")
print("-" * 100)
for prog, (query, db, use) in programs.items():
    print(f"{prog:<10} {query:<30} {db:<30} {use}")
print("\nProceed to Section 1.")

---

## 2. How BLAST Works: The Seed-and-Extend Heuristic

BLAST achieves its speed by **not** performing full Smith-Waterman on every query-database pair. Instead, it uses a three-phase heuristic:

### Phase 1: Find Seeds (Word Matching)

BLAST breaks the query into overlapping **words** of length $W$ (word size):
- Protein BLAST (blastp): $W = 3$ (default) — every 3-mer in the query
- Nucleotide BLAST (blastn): $W = 11$ (default) — every 11-mer

For protein BLAST, BLAST does not just look for exact word matches. It looks for **word neighborhoods** — all database words whose alignment score with the query word exceeds a threshold $T$. This is what makes blastp sensitive to divergent sequences even at the seeding stage.

```
Query word: LEW
Neighborhood (score >= T=11 with BLOSUM62):
  LEW (exact, score=14), LEF (score=12), LEY (score=11), LKW (score=11), ...

This allows BLAST to find seeds even when the database sequence differs slightly.
```

### Phase 2: Extend Seeds (Ungapped Extension)

When a seed is found, BLAST extends it in both directions **without allowing gaps**. Extension continues as long as the score does not drop more than $X$ below the best score seen so far (the **X-drop** heuristic). This produces a **high-scoring pair (HSP)**.

```
Query:    ...MVLSPADKTNVKAAWGKVGAHAG...
               |||||||||||||||||||||||
Database: ...MVLSGEDKSNIKAAWGKIGGHGAE...
                    ^ seed ^
              <-- extend left   extend right -->
              Stop when cumulative score drops X below maximum
```

### Phase 3: Gapped Extension (Full Alignment)

The highest-scoring ungapped HSPs are then extended with a **gapped alignment** (Smith-Waterman with dynamic programming). This is the expensive step, but by this point only a tiny fraction of database sequences are being processed.

### Why BLAST Is Approximate

BLAST may **miss** true homologs because:
1. No seed is found (query and target diverged too much in all word-length regions)
2. The X-drop terminates extension early

This is the sensitivity/speed tradeoff. PSI-BLAST and slower tools (e.g., SW search via SSEARCH) are more sensitive but slower.

### Summary of Key Parameters

| Parameter | Default | Effect |
|---|---|---|
| Word size ($W$) | 3 (protein), 11 (DNA) | Larger → faster, less sensitive |
| Word threshold ($T$) | 11 (blastp) | Higher → faster, less sensitive |
| Ungapped X-drop ($X_g$) | 20 | Higher → extends seeds further |
| Gapped X-drop ($X_u$) | 50 | Affects final extension |
| E-value cutoff | 10 | Higher → more hits reported |

In [ ]:
# Demonstration: word matching and neighborhood concept

from Bio.Align import substitution_matrices
import numpy as np

blosum62 = substitution_matrices.load("BLOSUM62")

def word_neighborhood(query_word, matrix, threshold=11):
    """
    Find all 3-letter amino acid words whose BLOSUM62 score with
    query_word exceeds the threshold T.
    
    This is what blastp does during the seeding phase.
    """
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    neighbors = []
    for a1 in amino_acids:
        for a2 in amino_acids:
            for a3 in amino_acids:
                word = a1 + a2 + a3
                score = (matrix[query_word[0], word[0]] +
                         matrix[query_word[1], word[1]] +
                         matrix[query_word[2], word[2]])
                if score >= threshold:
                    neighbors.append((word, score))
    return sorted(neighbors, key=lambda x: -x[1])


def xdrop_extend(seq1, seq2, match=2, mismatch=-1, x_drop=5):
    """
    Ungapped X-drop extension from the center of two sequences.
    Returns the score of the highest-scoring ungapped alignment.
    
    In real BLAST, this starts from a seed position and extends
    in both directions.
    """
    best_score = 0
    score = 0
    
    for i in range(min(len(seq1), len(seq2))):
        score += match if seq1[i] == seq2[i] else mismatch
        if score > best_score:
            best_score = score
        if best_score - score > x_drop:
            break
    return best_score


# Example 1: show word neighborhoods for "LEW"
print("=== BLAST Word Neighborhood (threshold T=11) ===")
query_word = "LEW"
neighbors = word_neighborhood(query_word, blosum62, threshold=11)
print(f"Query word: {query_word}")
print(f"Neighborhood size: {len(neighbors)} words")
print(f"Top 10 neighbors:")
for word, score in neighbors[:10]:
    print(f"  {word}: score = {score}")

# Example 2: compare neighborhood sizes for different thresholds
print()
print("=== Sensitivity vs Speed: Word Neighborhood Size vs Threshold ===")
print(f"{'Threshold T':>15} {'Neighborhood size':>20} {'Relative sensitivity'}")
print("-" * 55)
for T in [7, 9, 11, 13, 15]:
    n = len(word_neighborhood(query_word, blosum62, threshold=T))
    rel = n / len(word_neighborhood(query_word, blosum62, threshold=7)) * 100
    print(f"{T:>15} {n:>20} {rel:>18.1f}%")

print()
print("Higher T = smaller neighborhood = faster but less sensitive BLAST.")

# Example 3: X-drop extension demonstration
print()
print("=== X-drop Extension Demonstration ===")
seq_ref = "MVLSPADKTNVKAAWGKVGAHAG"
seq_hit = "MVLSGEDKSNIKAAWGKIGGHGAE"
score = xdrop_extend(seq_ref, seq_hit, match=2, mismatch=-1, x_drop=5)
print(f"Query:    {seq_ref}")
print(f"Database: {seq_hit}")
print(f"Ungapped X-drop score (X=5): {score}")
print()
print("The X-drop extension terminates when the running score drops more")
print("than X below the best score seen — avoiding wasted computation")
print("on sequences that diverge after a good seed.")

---

## 3. BLAST Program Variants

Different BLAST programs handle different combinations of query and database types:

| Program | Query | Database | Translation | Use Case |
|---|---|---|---|---|
| **blastn** | Nucleotide | Nucleotide | None | Gene finding, species ID, primer design |
| **blastp** | Protein | Protein | None | Protein homology, function prediction |
| **blastx** | Nucleotide | Protein | Query in 6 frames | EST annotation, finding coding regions |
| **tblastn** | Protein | Nucleotide | DB in 6 frames | Finding unannotated genes in genomes |
| **tblastx** | Nucleotide | Nucleotide | Both in 6 frames | Comparing unannotated genomes |

### Decision Guide

```
What is your QUERY?                 What is your DATABASE?

   DNA/RNA ----+----> DNA/RNA DB ---------> blastn (or megablast)
               |
               +----> Protein DB ---------> blastx (translate query)

   Protein ----+----> Protein DB ---------> blastp
               |
               +----> DNA/RNA DB ---------> tblastn (translate DB)

   DNA/RNA ----------> DNA/RNA DB ---------> tblastx (translate both)
   (when you suspect     (slow but most sensitive for
    distant homology)     distant nucleotide comparisons)
```

### Nucleotide BLAST Sub-variants

| Variant | Word size | Best for |
|---|---|---|
| **megablast** | 28 | Highly similar sequences (>95% identity), same species |
| **blastn** | 11 | Somewhat similar sequences, cross-species |
| **discontiguous megablast** | 11-12 | More dissimilar sequences (~80% identity) |

In [ ]:
# Helper to recommend BLAST program based on input
def recommend_blast(query_type, db_type, sensitivity="standard"):
    """Recommend the appropriate BLAST program."""
    recommendations = {
        ("nucleotide", "nucleotide"): {
            "high_identity": ("megablast", "Best for >95% identity, same species"),
            "standard": ("blastn", "Standard nucleotide search"),
            "sensitive": ("discontiguous megablast", "Cross-species, ~80% identity"),
            "very_sensitive": ("tblastx", "Both translated -- slowest but most sensitive"),
        },
        ("nucleotide", "protein"): {
            "standard": ("blastx", "Translates query in 6 frames"),
        },
        ("protein", "protein"): {
            "standard": ("blastp", "Standard protein search"),
            "sensitive": ("PSI-BLAST", "Iterative, for remote homologs"),
        },
        ("protein", "nucleotide"): {
            "standard": ("tblastn", "Translates DB in 6 frames"),
        },
    }
    
    key = (query_type, db_type)
    if key not in recommendations:
        return "Invalid combination"
    
    options = recommendations[key]
    if sensitivity in options:
        prog, desc = options[sensitivity]
    else:
        prog, desc = options["standard"]
    
    return f"{prog}: {desc}"


# Test various scenarios
scenarios = [
    ("nucleotide", "nucleotide", "high_identity", "Verify a clone sequence"),
    ("nucleotide", "nucleotide", "standard", "Find gene family members"),
    ("protein", "protein", "standard", "Find protein homologs"),
    ("protein", "protein", "sensitive", "Find distant protein homologs"),
    ("nucleotide", "protein", "standard", "Annotate an EST/mRNA"),
    ("protein", "nucleotide", "standard", "Find unannotated genes"),
]

print(f"{'Task':<35} {'Recommendation'}")
print("=" * 75)
for qt, dt, sens, task in scenarios:
    rec = recommend_blast(qt, dt, sens)
    print(f"{task:<35} {rec}")

---

## 4. Running BLAST via the NCBI Web Interface

The most accessible way to run BLAST is through the NCBI web server at: https://blast.ncbi.nlm.nih.gov/

### Step-by-step Guide

1. **Go to** https://blast.ncbi.nlm.nih.gov/
2. **Choose** your BLAST program (blastp, blastn, etc.)
3. **Enter** your query sequence (paste FASTA or accession number)
4. **Select** the database:
   - `nr` -- non-redundant (largest, most comprehensive)
   - `swissprot` -- manually curated (highest quality, smaller)
   - `pdb` -- sequences from protein structures
   - `refseq_protein` -- NCBI Reference Sequence proteins
5. **Adjust parameters** (usually defaults are fine):
   - E-value threshold (default: 10 -- intentionally permissive)
   - Word size
   - Scoring matrix (BLOSUM62 default)
   - Gap penalties
6. **Submit** and wait for results (usually 10-60 seconds)

### Reading the Results Page

The NCBI results page has several sections:

| Section | What it shows |
|---|---|
| **Graphic summary** | Visual map of where hits align to your query |
| **Descriptions** | Table of hits sorted by E-value |
| **Alignments** | Detailed pairwise alignments for each hit |
| **Taxonomy** | Taxonomic distribution of hits |

You can download results in various formats: XML, tabular (TSV), JSON, or plain text.

---

## 5. Running BLAST Programmatically with BioPython

For reproducible research and batch processing, we run BLAST from Python using `Bio.Blast.NCBIWWW`.

**Important**: NCBI requests that you:
- Do not submit more than one request every 10 seconds
- Run large jobs during off-peak hours (US evenings/weekends)
- Use local BLAST for high-throughput work

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import SeqIO


def run_blast(sequence, program="blastp", database="swissprot",
              evalue=0.001, hitlist_size=20, word_size=None):
    """Run a BLAST search against NCBI and return parsed results.
    
    Parameters
    ----------
    sequence : str
        Query sequence (protein or nucleotide).
    program : str
        BLAST program: blastn, blastp, blastx, tblastn, tblastx.
    database : str
        Target database: nr, swissprot, nt, pdb, refseq_protein, etc.
    evalue : float
        E-value threshold.
    hitlist_size : int
        Maximum number of hits to return.
    word_size : int or None
        Word size (None = use default for the program).
    
    Returns
    -------
    Bio.Blast.Record
        Parsed BLAST record.
    """
    print(f"Running {program} against {database} (this may take 30-120 seconds)...")
    
    kwargs = dict(
        program=program,
        database=database,
        sequence=sequence,
        expect=evalue,
        hitlist_size=hitlist_size,
    )
    if word_size is not None:
        kwargs["word_size"] = word_size
    
    result_handle = NCBIWWW.qblast(**kwargs)
    blast_record = NCBIXML.read(result_handle)
    result_handle.close()
    
    print(f"Done. Found {len(blast_record.alignments)} hits.")
    return blast_record


print("run_blast() function defined.")
print("\nUsage:")
print('  record = run_blast("MVLSPADKTNVK...", "blastp", "swissprot")')

In [ ]:
# Example: search human hemoglobin alpha against SwissProt
# Uncomment the next lines to run (requires internet, takes ~1 minute)

hba_human = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLL"
    "SHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
)

# blast_record = run_blast(hba_human, "blastp", "swissprot", hitlist_size=15)

print(f"Query: Human Hemoglobin Alpha ({len(hba_human)} aa)")
print("Uncomment the line above to run the actual BLAST search.")

---

## 6. Parsing BLAST Results

BLAST results have a hierarchical structure:

```
BLAST Record
  +-- query info (length, description)
  +-- database info (name, size)
  +-- Alignment 1 (= one database hit)
  |     +-- HSP 1 (High-scoring Segment Pair)
  |     +-- HSP 2 (if multiple distinct regions match)
  +-- Alignment 2
  |     +-- HSP 1
  +-- ...
```

### Key Fields in an HSP

| Field | Description |
|---|---|
| `hsp.score` | Raw alignment score |
| `hsp.bits` | Bit score (normalized) |
| `hsp.expect` | E-value |
| `hsp.identities` | Number of identical positions |
| `hsp.positives` | Number of positive-scoring positions (identities + conservative) |
| `hsp.gaps` | Number of gap positions |
| `hsp.align_length` | Alignment length |
| `hsp.query` | Aligned query sequence |
| `hsp.sbjct` | Aligned subject sequence |
| `hsp.match` | Match line (showing identities and positives) |

In [ ]:
import pandas as pd


def blast_results_to_dataframe(blast_record):
    """Convert BLAST results to a pandas DataFrame for easy analysis."""
    rows = []
    for alignment in blast_record.alignments:
        for hsp in alignment.hsps:
            identity_pct = 100 * hsp.identities / hsp.align_length
            coverage = 100 * (hsp.query_end - hsp.query_start + 1) / blast_record.query_length
            
            rows.append({
                "hit_id": alignment.hit_id,
                "description": alignment.hit_def[:70],
                "hit_length": alignment.length,
                "e_value": hsp.expect,
                "bit_score": round(hsp.bits, 1),
                "raw_score": hsp.score,
                "identity_pct": round(identity_pct, 1),
                "positives_pct": round(100 * hsp.positives / hsp.align_length, 1),
                "gaps": hsp.gaps,
                "align_length": hsp.align_length,
                "coverage_pct": round(coverage, 1),
                "query_start": hsp.query_start,
                "query_end": hsp.query_end,
                "sbjct_start": hsp.sbjct_start,
                "sbjct_end": hsp.sbjct_end,
            })
    
    return pd.DataFrame(rows)


print("blast_results_to_dataframe() defined.")
print("\nUsage:")
print("  df = blast_results_to_dataframe(blast_record)")
print("  df.head()")

In [ ]:
def print_blast_summary(blast_record, max_hits=10):
    """Print a formatted summary of BLAST results."""
    print(f"Query: {blast_record.query[:60]}")
    print(f"Query length: {blast_record.query_length}")
    print(f"Database: {blast_record.database}")
    print(f"Total hits: {len(blast_record.alignments)}")
    print()
    print(f"{'#':<3} {'Description':<50} {'E-value':>10} {'Bit':>6} {'Id%':>6} {'Cov%':>6}")
    print("-" * 85)
    
    for i, aln in enumerate(blast_record.alignments[:max_hits]):
        hsp = aln.hsps[0]  # best HSP
        id_pct = 100 * hsp.identities / hsp.align_length
        cov = 100 * (hsp.query_end - hsp.query_start + 1) / blast_record.query_length
        desc = aln.hit_def[:47] + "..." if len(aln.hit_def) > 50 else aln.hit_def
        print(f"{i+1:<3} {desc:<50} {hsp.expect:>10.1e} {hsp.bits:>6.0f} {id_pct:>5.1f}% {cov:>5.1f}%")


def print_blast_alignment(alignment, hsp_index=0):
    """Print a formatted BLAST alignment (similar to NCBI output)."""
    hsp = alignment.hsps[hsp_index]
    print(f"> {alignment.hit_def[:80]}")
    print(f"  Length = {alignment.length}")
    print(f"  Score = {hsp.bits:.0f} bits ({hsp.score}), E-value = {hsp.expect:.1e}")
    print(f"  Identities = {hsp.identities}/{hsp.align_length} "
          f"({100*hsp.identities/hsp.align_length:.0f}%), "
          f"Positives = {hsp.positives}/{hsp.align_length} "
          f"({100*hsp.positives/hsp.align_length:.0f}%), "
          f"Gaps = {hsp.gaps}/{hsp.align_length}")
    print()
    
    # Print alignment in blocks of 60
    width = 60
    q_pos = hsp.query_start
    s_pos = hsp.sbjct_start
    for i in range(0, len(hsp.query), width):
        q_seg = hsp.query[i:i+width]
        m_seg = hsp.match[i:i+width]
        s_seg = hsp.sbjct[i:i+width]
        q_end = q_pos + len(q_seg.replace("-", "")) - 1
        s_end = s_pos + len(s_seg.replace("-", "")) - 1
        print(f"  Query {q_pos:>5}  {q_seg}  {q_end}")
        print(f"               {m_seg}")
        print(f"  Sbjct {s_pos:>5}  {s_seg}  {s_end}")
        print()
        q_pos = q_end + 1
        s_pos = s_end + 1


print("Summary and alignment printing functions defined.")

### Working with Pre-computed Results

Since BLAST web searches take time, let us work with saved XML results. You can save any BLAST result by downloading the XML from NCBI.

In [ ]:
# Simulating BLAST-like results for demonstration
# In practice, you would load saved XML:
#   with open("blast_results.xml") as f:
#       blast_record = NCBIXML.read(f)

# Let us create a mock record to demonstrate parsing
class MockHSP:
    def __init__(self, expect, bits, score, identities, positives, gaps, 
                 align_length, query, sbjct, match, 
                 query_start, query_end, sbjct_start, sbjct_end):
        self.expect = expect
        self.bits = bits
        self.score = score
        self.identities = identities
        self.positives = positives
        self.gaps = gaps
        self.align_length = align_length
        self.query = query
        self.sbjct = sbjct
        self.match = match
        self.query_start = query_start
        self.query_end = query_end
        self.sbjct_start = sbjct_start
        self.sbjct_end = sbjct_end


class MockAlignment:
    def __init__(self, hit_id, hit_def, length, hsps):
        self.hit_id = hit_id
        self.hit_def = hit_def
        self.length = length
        self.hsps = hsps


class MockBlastRecord:
    def __init__(self, query, query_length, database, alignments):
        self.query = query
        self.query_length = query_length
        self.database = database
        self.alignments = alignments


# Create realistic mock results for hemoglobin alpha
mock_alignments = [
    MockAlignment("sp|P69905|HBA_HUMAN", "Hemoglobin subunit alpha OS=Homo sapiens", 142,
        [MockHSP(2e-99, 289, 740, 141, 141, 0, 142,
                 "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 1, 142, 1, 142)]),
    MockAlignment("sp|P01942|HBA_MOUSE", "Hemoglobin subunit alpha OS=Mus musculus", 142,
        [MockHSP(1e-85, 252, 643, 120, 132, 0, 142,
                 "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH",
                 "MVLS  DK N KAAWGK G H  EYGAEALERMF SFPTTKTYFPHFD SH",
                 1, 142, 1, 142)]),
    MockAlignment("sp|P01958|HBA_CHICK", "Hemoglobin subunit alpha-A OS=Gallus gallus", 142,
        [MockHSP(3e-68, 205, 522, 98, 118, 0, 142,
                 "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVLSAADKNNVKGIFTKIAGHAEEYGAETLERMFTTYPPTKTYFPHFDLSH",
                 "MVLS  DK NVK  + K  GHA EYGAE LERMF  +PPTKTYFPHFDLSH",
                 1, 142, 1, 142)]),
    MockAlignment("sp|P02024|HBB_HUMAN", "Hemoglobin subunit beta OS=Homo sapiens", 147,
        [MockHSP(5e-25, 98, 244, 44, 67, 3, 147,
                 "MVLSPADKTNVK-AAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVHLSAEEKAAVTAFWGKVKVDEVGGEALGRLLVVYPWTQRFFESFGDLSS",
                 "MV +S  +K  V A WGKV    +G EA  +L  + P T+ F  F  LS ",
                 1, 141, 1, 147)]),
    MockAlignment("sp|P02185|MYG_PHYCA", "Myoglobin OS=Physeter catodon", 154,
        [MockHSP(2e-15, 65, 157, 35, 55, 5, 155,
                 "MVLSPADKTNVKAAWGKVGAHAG-EYGAEALERMFLSFPTTKTYFPHFDLSH",
                 "MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLK",
                 "MVL+  +  +V  +W K     A H  + +L RL  S P T  +F  F  L ",
                 1, 141, 1, 154)]),
]

mock_record = MockBlastRecord(
    "HBA_HUMAN Hemoglobin subunit alpha", 142, "swissprot", mock_alignments
)

print("Mock BLAST record created (simulating hemoglobin alpha search).")

In [ ]:
# Display the summary table
print_blast_summary(mock_record)

In [ ]:
# Show a detailed alignment for the mouse hit
print_blast_alignment(mock_record.alignments[1])

In [ ]:
# Convert to DataFrame for analysis
df = blast_results_to_dataframe(mock_record)
print(df[["hit_id", "e_value", "bit_score", "identity_pct", "coverage_pct"]].to_string(index=False))

### Parsing Tabular BLAST Output (outfmt 6)

When running local BLAST, tabular output (`-outfmt 6`) is often the most convenient for programmatic parsing. It produces tab-separated values with no header.

In [ ]:
# Standard tabular BLAST columns (outfmt 6)
BLAST_TABULAR_COLUMNS = [
    "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore"
]

# Example tabular output (what you would get from: blastp -outfmt 6)
example_tabular = """HBA_HUMAN	sp|P69905|HBA_HUMAN	100.000	142	0	0	1	142	1	142	2.00e-99	289.0
HBA_HUMAN	sp|P01942|HBA_MOUSE	84.507	142	22	0	1	142	1	142	1.00e-85	252.0
HBA_HUMAN	sp|P01958|HBA_CHICK	69.014	142	44	0	1	142	1	142	3.00e-68	205.0
HBA_HUMAN	sp|P02024|HBB_HUMAN	29.932	147	100	3	1	141	1	147	5.00e-25	98.0
HBA_HUMAN	sp|P02185|MYG_PHYCA	22.581	155	115	5	1	141	1	154	2.00e-15	65.0"""

# Parse with pandas
from io import StringIO

df_tabular = pd.read_csv(StringIO(example_tabular), sep="\t",
                          names=BLAST_TABULAR_COLUMNS)

print("Parsed tabular BLAST output:")
print(df_tabular.to_string(index=False))

---

## 7. Understanding BLAST Statistics

### 7.1 E-value (Expect Value)

The E-value answers: **"How many alignments with this score or better would I expect to find by chance in a database of this size?"**

$$E = K \cdot m \cdot n \cdot e^{-\lambda S}$$

where:
- $m$ = query length (effective)
- $n$ = total database length (effective)
- $S$ = raw alignment score
- $K$, $\lambda$ = Karlin-Altschul statistical parameters

**Key properties:**
- E-value is NOT a probability (it can be > 1)
- E-value depends on database size: same alignment has different E-values in different databases
- E-value = 0.01 means we expect 0.01 such alignments by chance (i.e., 1 in 100 searches)

### 7.2 Bit Score

$$S' = \frac{\lambda S - \ln K}{\ln 2}$$

Bit scores are **normalized** and **database-independent**. A bit score of 50 means the alignment is $2^{50}$ times more likely than expected by chance.

### 7.3 Percent Identity vs Query Coverage

Both matter for assessing homology:

| Scenario | Identity | Coverage | Interpretation |
|---|---|---|---|
| Full-length match | 95% | 100% | Very close homolog (ortholog?) |
| Full-length match | 35% | 95% | Distant homolog, same family |
| Partial match | 95% | 20% | Shared domain, different architecture |
| Partial match | 25% | 15% | Probably not significant |

In [ ]:
import matplotlib.pyplot as plt

# Visualize E-value vs bit score relationship
bit_scores = np.linspace(20, 200, 100)
query_len = 300
db_sizes = {
    "SwissProt (2e8 residues)": 2e8,
    "nr (9e10 residues)": 9e10,
}

fig, ax = plt.subplots(figsize=(9, 5))
for name, n in db_sizes.items():
    e_values = query_len * n * 2**(-bit_scores)
    ax.semilogy(bit_scores, e_values, label=name, linewidth=2)

ax.axhline(0.001, color="red", linestyle="--", alpha=0.5, label="Typical threshold (E=0.001)")
ax.axhline(1e-10, color="green", linestyle="--", alpha=0.5, label="Stringent threshold (E=1e-10)")
ax.set_xlabel("Bit score")
ax.set_ylabel("E-value")
ax.set_title("E-value vs Bit Score for Different Database Sizes")
ax.legend()
ax.set_ylim(1e-100, 1e10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize our mock BLAST results
import math

df = blast_results_to_dataframe(mock_record)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: E-value distribution
ax = axes[0]
log_evalues = [-math.log10(e) if e > 0 else 300 for e in df["e_value"]]
ax.barh(range(len(df)), log_evalues, color="steelblue")
ax.set_yticks(range(len(df)))
ax.set_yticklabels([h.split("|")[-1] if "|" in h else h[:15] for h in df["hit_id"]])
ax.set_xlabel("-log10(E-value)")
ax.set_title("Significance")
ax.invert_yaxis()

# Plot 2: Identity %
ax = axes[1]
colors = ["darkgreen" if x > 50 else "orange" if x > 30 else "red" for x in df["identity_pct"]]
ax.barh(range(len(df)), df["identity_pct"], color=colors)
ax.set_yticks(range(len(df)))
ax.set_yticklabels([h.split("|")[-1] if "|" in h else h[:15] for h in df["hit_id"]])
ax.set_xlabel("Identity (%)")
ax.set_title("Percent Identity")
ax.axvline(30, color="red", linestyle="--", alpha=0.5, label="Twilight zone")
ax.legend(fontsize=8)
ax.invert_yaxis()

# Plot 3: Bit score
ax = axes[2]
ax.barh(range(len(df)), df["bit_score"], color="coral")
ax.set_yticks(range(len(df)))
ax.set_yticklabels([h.split("|")[-1] if "|" in h else h[:15] for h in df["hit_id"]])
ax.set_xlabel("Bit score")
ax.set_title("Bit Score")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print("Observations:")
print("- HBA_HUMAN (self-hit): perfect match")
print("- HBA_MOUSE: close ortholog (84.5% identity, E ~ 1e-85)")
print("- HBA_CHICK: more distant ortholog (69% identity)")
print("- HBB_HUMAN: paralog (different subunit, 30% identity)")
print("- MYG_PHYCA: remote globin family member (23% identity -- twilight zone!)")

---

## 8. PSI-BLAST: Finding Remote Homologs

**Position-Specific Iterated BLAST** (PSI-BLAST) is designed to find **distant homologs** that standard BLAST may miss.

### How PSI-BLAST Works

1. **Iteration 1**: Run standard blastp and collect significant hits
2. **Build PSSM**: Construct a Position-Specific Scoring Matrix from the alignment of significant hits
3. **Iteration 2**: Search with the PSSM instead of the raw sequence -- this captures position-specific conservation patterns
4. **Repeat**: Include newly found significant hits, rebuild PSSM, search again
5. **Converge**: Stop when no new significant hits are found

```
Iteration 1: Standard BLASTP
  Query: MVLSPADKTNVKAAWGKVGAHAG...
  Hits:  Close homologs (E < 0.001)
                   |
                   v
Iteration 2: Build PSSM from hits
  PSSM captures: "Position 5 must be P or S"
                 "Position 15 is always aromatic"
  New hits: More distant homologs found!
                   |
                   v
Iteration 3: Refined PSSM
  Even more distant homologs...
                   |
                   v
Convergence: No new hits found
```

### When to Use PSI-BLAST

- When standard blastp finds few or no significant hits
- When you suspect the protein belongs to a large, diverse family
- When searching for structural homologs (fold recognition)

### Dangers of PSI-BLAST

- **Profile corruption**: If a non-homologous sequence sneaks in during early iterations, it can corrupt the PSSM and lead to a cascade of false positives
- **Mitigation**: Use stringent E-value thresholds for inclusion (e.g., 0.001); manually check borderline hits

In [ ]:
# PSI-BLAST concept demonstration: building a PSSM from aligned sequences

# Simulated alignment of hemoglobin alpha orthologs (first 20 positions)
aligned_sequences = [
    "MVLSPADKTNVKAAWGKVGA",  # Human
    "MVLSGEDKSNIKAAWGKIGG",  # Mouse
    "MVLSAADKNNVKGIFTKIAG",  # Chicken
    "MVLSAADKTNVKAAFSKVAG",  # Horse
    "MVLSANDKTNVKAAWSKVGG",  # Pig
]

def build_simple_pssm(sequences, pseudocount=1):
    """Build a simple frequency-based PSSM from aligned sequences."""
    n_seqs = len(sequences)
    length = len(sequences[0])
    aa_order = "ACDEFGHIKLMNPQRSTVWY"
    
    pssm = np.zeros((length, len(aa_order)))
    
    for pos in range(length):
        for seq in sequences:
            if seq[pos] in aa_order:
                idx = aa_order.index(seq[pos])
                pssm[pos, idx] += 1
        # Add pseudocounts and normalize
        pssm[pos] = (pssm[pos] + pseudocount) / (n_seqs + pseudocount * len(aa_order))
    
    return pssm, aa_order


pssm, aa_order = build_simple_pssm(aligned_sequences)

# Show the PSSM for first 10 positions
print("Position-Specific Scoring Matrix (frequencies):")
print(f"{'Pos':>4} {'Res':>4}  Most likely residues")
print("-" * 50)
for pos in range(min(20, len(aligned_sequences[0]))):
    consensus = aligned_sequences[0][pos]
    # Top 3 amino acids at this position
    top_indices = np.argsort(pssm[pos])[::-1][:3]
    top_aas = [(aa_order[i], pssm[pos, i]) for i in top_indices if pssm[pos, i] > 0.05]
    top_str = ", ".join(f"{aa}({freq:.2f})" for aa, freq in top_aas)
    conservation = max(pssm[pos])
    bar = "*" * int(conservation * 20)
    print(f"{pos+1:>4} {consensus:>4}  {top_str:<40} {bar}")

In [ ]:
# Visualize the PSSM as a heatmap (sequence logo-like)
fig, ax = plt.subplots(figsize=(14, 6))

# Only show the most informative amino acids
im = ax.imshow(pssm.T, cmap="YlOrRd", aspect="auto", interpolation="none")
ax.set_yticks(range(len(aa_order)))
ax.set_yticklabels(list(aa_order), fontsize=8)
ax.set_xlabel("Position in alignment")
ax.set_ylabel("Amino acid")
ax.set_title("PSSM Heatmap -- Hemoglobin Alpha (first 20 positions)")
plt.colorbar(im, label="Frequency")

# Mark the consensus residue at each position
for pos in range(pssm.shape[0]):
    best_aa = aa_order[np.argmax(pssm[pos])]
    best_idx = np.argmax(pssm[pos])
    ax.text(pos, best_idx, best_aa, ha="center", va="center", 
            fontsize=7, fontweight="bold", color="blue")

plt.tight_layout()
plt.show()

print("PSI-BLAST uses this kind of position-specific information")
print("instead of a single substitution matrix at every position.")

---

## 9. Best Practices for BLAST Searching

### 9.1 Database Selection

| Database | Size | Quality | Best for |
|---|---|---|---|
| **nr** (non-redundant) | Largest | Mixed | Comprehensive search |
| **swissprot** | ~570K seqs | High (curated) | Reliable annotations |
| **pdb** | ~200K seqs | High (structures known) | Structural homologs |
| **refseq_protein** | Large | High | Reference annotations |
| **nt** | Very large | Mixed | Nucleotide searches |
| **refseq_genomic** | Large | High | Genome-level searches |

**Rule of thumb**: Start with SwissProt for protein searches (fast, high-quality annotations). If you need more hits, expand to nr.

### 9.2 Parameter Tuning

| Parameter | Default | Tune when... |
|---|---|---|
| **E-value** | 10 | Lower (0.001) for stringent searches; higher for remote homologs |
| **Word size** | 3 (protein) / 11 (DNA) | Decrease for more sensitivity; increase for speed |
| **Matrix** | BLOSUM62 | BLOSUM45 for distant sequences; BLOSUM80 for close ones |
| **Gap open/extend** | -11/-1 | Experiment if alignments have unreasonable gaps |
| **Low-complexity filter** | ON (SEG/DUST) | Turn OFF only if query has real low-complexity regions |
| **Composition-based stats** | ON | Reduces false positives from compositional bias |

### 9.3 Common Mistakes

1. **Using the wrong BLAST program**: Using blastn when blastx would be better for finding coding regions
2. **Not filtering low-complexity regions**: Poly-A tails, proline-rich regions, etc. cause false positives
3. **Using default E-value (10) without awareness**: This is intentionally permissive -- many hits will be non-significant
4. **Trusting percent identity alone**: 90% identity over 10 residues is meaningless; 30% over 200 residues is significant
5. **Ignoring query coverage**: A hit that covers only 5% of your query may be a shared domain, not a full homolog

In [ ]:
# Demonstrate why low-complexity filtering matters

def sequence_complexity(seq, window=20):
    """Calculate Shannon entropy in sliding windows as a measure of complexity."""
    from collections import Counter
    import math
    
    entropies = []
    for i in range(len(seq) - window + 1):
        w = seq[i:i+window]
        counts = Counter(w)
        total = sum(counts.values())
        entropy = -sum((c/total) * math.log2(c/total) for c in counts.values())
        entropies.append(entropy)
    return entropies


# Normal protein sequence
normal = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
# Sequence with a low-complexity region (proline-rich)
low_complex = "MVLSPADKPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPTTKTYFPHFDLSH"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, seq, title in zip(axes, [normal, low_complex],
                          ["Normal sequence", "Low-complexity region"]):
    entropies = sequence_complexity(seq, window=10)
    ax.plot(entropies, color="steelblue")
    ax.set_xlabel("Position")
    ax.set_ylabel("Shannon entropy (bits)")
    ax.set_title(title)
    ax.set_ylim(0, 4.5)
    ax.axhline(2.0, color="red", linestyle="--", alpha=0.5, label="Low complexity threshold")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Low-complexity regions (like poly-P, poly-Q, or collagen-like GXY repeats)")
print("produce spurious BLAST hits. The SEG filter masks these regions with 'X' characters.")

### 9.4 Local BLAST Installation

For high-throughput work, install BLAST+ locally:

```bash
# macOS
brew install blast

# Ubuntu/Debian
sudo apt install ncbi-blast+

# Conda
conda install -c bioconda blast

# Verify
blastp -version
```

### Creating a local database

```bash
# From a FASTA file
makeblastdb -in my_proteins.fasta -dbtype prot -out my_db -parse_seqids

# Run search
blastp -query query.fasta -db my_db -out results.txt \
       -outfmt 6 -evalue 0.001 -num_threads 4

# Custom tabular columns
blastp -query query.fasta -db my_db \
       -outfmt "6 qseqid sseqid pident evalue bitscore stitle" \
       -evalue 0.001
```

In [ ]:
# BioPython wrapper for local BLAST (NcbiblastpCommandline)
from Bio.Blast.Applications import NcbiblastpCommandline

# Build the command (not executed -- requires local BLAST installation)
cmd = NcbiblastpCommandline(
    query="query.fasta",
    db="swissprot",
    evalue=0.001,
    outfmt=5,          # XML output
    out="results.xml",
    num_threads=4,
    max_target_seqs=50,
)

print("Generated command:")
print(str(cmd))
print("\nTo run: cmd() or execute the command string in a terminal.")
print("Parse results: NCBIXML.parse(open('results.xml'))")

---

## 10. Putting It All Together: A BLAST Analysis Workflow

Here is a complete workflow for analyzing a mystery protein sequence.

In [ ]:
def blast_analysis_workflow(sequence, name="unknown"):
    """Complete BLAST analysis workflow for a protein sequence.
    
    This function demonstrates the full analysis pipeline.
    The actual BLAST search is commented out to avoid network calls.
    """
    print(f"=" * 70)
    print(f"BLAST Analysis Workflow for: {name}")
    print(f"=" * 70)
    
    # Step 1: Validate the sequence
    print(f"\n1. SEQUENCE VALIDATION")
    print(f"   Length: {len(sequence)} residues")
    aa_set = set(sequence.upper())
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    invalid = aa_set - valid_aa
    if invalid:
        print(f"   WARNING: Non-standard characters found: {invalid}")
    else:
        print(f"   All characters are standard amino acids.")
    
    # Step 2: Check for low-complexity
    print(f"\n2. COMPLEXITY CHECK")
    entropies = sequence_complexity(sequence, window=12)
    if entropies:
        avg_entropy = np.mean(entropies)
        min_entropy = min(entropies)
        print(f"   Average entropy: {avg_entropy:.2f} bits")
        print(f"   Minimum entropy: {min_entropy:.2f} bits")
        if min_entropy < 1.5:
            print(f"   WARNING: Low-complexity region detected. Enable SEG filtering.")
        else:
            print(f"   No low-complexity regions detected.")
    
    # Step 3: Choose BLAST parameters
    print(f"\n3. RECOMMENDED PARAMETERS")
    print(f"   Program: blastp")
    if len(sequence) < 30:
        print(f"   Matrix: PAM30 (short query)")
        print(f"   Word size: 2")
    elif len(sequence) < 80:
        print(f"   Matrix: BLOSUM62")
        print(f"   Word size: 3")
    else:
        print(f"   Matrix: BLOSUM62 (standard)")
        print(f"   Word size: 6 (fast) or 3 (sensitive)")
    print(f"   Database: swissprot (start here, then nr if needed)")
    print(f"   E-value: 0.001")
    
    # Step 4: BLAST search (Ready)
    print(f"\n4. BLAST SEARCH")
    print(f"   # Uncomment to run:")
    print(f"   # record = run_blast(sequence, 'blastp', 'swissprot')")
    print(f"   # df = blast_results_to_dataframe(record)")
    print(f"   # print_blast_summary(record)")
    
    # Step 5: Interpretation guide
    print(f"\n5. INTERPRETATION GUIDE")
    print(f"   - E < 1e-10: Confident homologs")
    print(f"   - Identity > 30%: Above twilight zone (for proteins)")
    print(f"   - Coverage > 70%: Full-length match (not just a domain)")
    print(f"   - Check if top hits agree on function/family")
    print(f"   - If few hits: try PSI-BLAST for remote homologs")


# Run the workflow
mystery_sequence = (
    "PQITLWQRPLVTIRIGGQLKEALLDTGADDTVLEEMNLPGKWKPKMIGGIGGFIKVRQYD"
    "QIPVEIAHKAIGTVLVGPTPVNIIGRNLLTQIGATL"
)

blast_analysis_workflow(mystery_sequence, "Mystery Protein")

In [ ]:
# Interpreting results: a decision framework

def interpret_blast_hit(e_value, identity_pct, coverage_pct, alignment_length):
    """Provide an interpretation of a BLAST hit."""
    print(f"  E-value:     {e_value:.1e}")
    print(f"  Identity:    {identity_pct:.1f}%")
    print(f"  Coverage:    {coverage_pct:.1f}%")
    print(f"  Aln length:  {alignment_length}")
    print()
    
    # E-value assessment
    if e_value < 1e-50:
        e_verdict = "Extremely significant -- virtually identical"
    elif e_value < 1e-10:
        e_verdict = "Very significant -- clear homolog"
    elif e_value < 0.001:
        e_verdict = "Significant -- probable homolog"
    elif e_value < 1:
        e_verdict = "Marginal -- possible homology, needs verification"
    else:
        e_verdict = "Not significant -- likely random match"
    
    # Identity assessment (protein-specific)
    if identity_pct > 50:
        id_verdict = "High identity -- close homolog"
    elif identity_pct > 30:
        id_verdict = "Moderate identity -- same protein family"
    elif identity_pct > 20:
        id_verdict = "Twilight zone -- homology uncertain"
    else:
        id_verdict = "Below twilight zone -- cannot confirm homology by identity alone"
    
    # Coverage assessment
    if coverage_pct > 80:
        cov_verdict = "Full-length match"
    elif coverage_pct > 50:
        cov_verdict = "Partial match -- may share some domains"
    else:
        cov_verdict = "Low coverage -- possibly just a shared domain or motif"
    
    print(f"  Assessment:")
    print(f"    E-value:  {e_verdict}")
    print(f"    Identity: {id_verdict}")
    print(f"    Coverage: {cov_verdict}")


# Test with our mock results
test_cases = [
    ("Close ortholog (Mouse HBA)", 1e-85, 84.5, 100.0, 142),
    ("Distant ortholog (Chicken HBA)", 3e-68, 69.0, 100.0, 142),
    ("Paralog (Human HBB)", 5e-25, 29.9, 99.3, 147),
    ("Remote homolog (Myoglobin)", 2e-15, 22.6, 99.3, 155),
]

for name, e, ident, cov, length in test_cases:
    print(f"\n--- {name} ---")
    interpret_blast_hit(e, ident, cov, length)

### Reciprocal Best BLAST Hits (RBH)

A powerful strategy for identifying **orthologs** (genes in different species descended from a common ancestor) is the Reciprocal Best Hit approach:

1. BLAST protein A from species 1 against all proteins of species 2
2. Take the top hit (protein B)
3. BLAST protein B back against all proteins of species 1
4. If the top hit is protein A, they are **reciprocal best hits** -- strong evidence for orthology

```
Species 1                     Species 2
                  BLAST -->
  Protein A  ==================> Protein B  (top hit)
  Protein A  <================== Protein B  (top hit)
                  <-- BLAST

  If both arrows point to each other => Reciprocal Best Hit (likely orthologs)
```

RBH is not perfect -- it can fail with gene duplications (paralogs) -- but it is a quick, widely used heuristic.

In [ ]:
def find_reciprocal_best_hits(blast_forward, blast_reverse):
    """Identify reciprocal best hits from two BLAST result DataFrames.
    
    Parameters
    ----------
    blast_forward : pd.DataFrame
        BLAST results: species1 queries vs species2 database.
        Must have columns: 'qseqid', 'sseqid', 'evalue', 'bitscore'
    blast_reverse : pd.DataFrame
        BLAST results: species2 queries vs species1 database.
    
    Returns
    -------
    list of tuple
        Pairs (species1_id, species2_id) that are reciprocal best hits.
    """
    # Best hit for each query in forward search
    fwd_best = blast_forward.sort_values('evalue').groupby('qseqid').first()
    # Best hit for each query in reverse search
    rev_best = blast_reverse.sort_values('evalue').groupby('qseqid').first()
    
    rbh_pairs = []
    for q1, row in fwd_best.iterrows():
        best_in_sp2 = row['sseqid']
        # Check if that species2 gene's best hit back is q1
        if best_in_sp2 in rev_best.index:
            best_back = rev_best.loc[best_in_sp2, 'sseqid']
            if best_back == q1:
                rbh_pairs.append((q1, best_in_sp2))
    
    return rbh_pairs


# Demo with mock data
fwd = pd.DataFrame({
    'qseqid': ['HBA_HUMAN', 'HBA_HUMAN', 'HBB_HUMAN', 'HBB_HUMAN'],
    'sseqid': ['HBA_MOUSE', 'HBB_MOUSE', 'HBB_MOUSE', 'HBA_MOUSE'],
    'evalue': [1e-85, 5e-25, 1e-90, 3e-24],
    'bitscore': [252, 98, 270, 95],
})
rev = pd.DataFrame({
    'qseqid': ['HBA_MOUSE', 'HBA_MOUSE', 'HBB_MOUSE', 'HBB_MOUSE'],
    'sseqid': ['HBA_HUMAN', 'HBB_HUMAN', 'HBB_HUMAN', 'HBA_HUMAN'],
    'evalue': [1e-85, 4e-24, 1e-90, 5e-25],
    'bitscore': [252, 96, 270, 98],
})

rbh = find_reciprocal_best_hits(fwd, rev)
print("Reciprocal Best Hits:")
for a, b in rbh:
    print(f"  {a} <--> {b}")
print(f"\nThese pairs are candidate orthologs.")

### Filtering and Ranking BLAST Results

In practice, raw BLAST output needs filtering before biological conclusions can be drawn. Here is a standard filtering pipeline.

In [ ]:
def filter_blast_results(df, max_evalue=1e-5, min_identity=25.0,
                         min_coverage=50.0, min_bitscore=50):
    """Apply standard filters to BLAST results.
    
    Returns filtered DataFrame and prints a summary of what was removed.
    """
    n_total = len(df)
    
    # Apply filters stepwise
    df_f = df[df['e_value'] <= max_evalue]
    n_after_eval = len(df_f)
    
    df_f = df_f[df_f['identity_pct'] >= min_identity]
    n_after_id = len(df_f)
    
    if 'coverage_pct' in df_f.columns:
        df_f = df_f[df_f['coverage_pct'] >= min_coverage]
    n_after_cov = len(df_f)
    
    df_f = df_f[df_f['bit_score'] >= min_bitscore]
    n_after_bit = len(df_f)
    
    print(f"Filtering summary:")
    print(f"  Total hits:         {n_total}")
    print(f"  After E-value:      {n_after_eval} (removed {n_total - n_after_eval})")
    print(f"  After identity:     {n_after_id} (removed {n_after_eval - n_after_id})")
    print(f"  After coverage:     {n_after_cov} (removed {n_after_id - n_after_cov})")
    print(f"  After bit score:    {n_after_bit} (removed {n_after_cov - n_after_bit})")
    print(f"  Final: {n_after_bit}/{n_total} hits retained")
    
    return df_f


# Apply to our mock results
df_all = blast_results_to_dataframe(mock_record)
df_filtered = filter_blast_results(df_all, max_evalue=1e-10, min_identity=25.0)

### Visualizing BLAST Hit Coverage Along the Query

An important way to understand BLAST results is to visualize where each hit aligns along the query sequence -- similar to the graphical overview on the NCBI BLAST results page.

In [ ]:
def plot_blast_coverage(blast_record, max_hits=10):
    """Visualize BLAST hit positions along the query (NCBI-style graphic)."""
    import matplotlib.patches as mpatches
    
    fig, ax = plt.subplots(figsize=(12, max(3, max_hits * 0.4)))
    
    query_len = blast_record.query_length
    
    for i, aln in enumerate(blast_record.alignments[:max_hits]):
        hsp = aln.hsps[0]
        y = i
        
        # Color by E-value
        import math
        if hsp.expect == 0:
            color = "darkred"
        elif hsp.expect < 1e-50:
            color = "red"
        elif hsp.expect < 1e-10:
            color = "orange"
        elif hsp.expect < 0.01:
            color = "green"
        else:
            color = "gray"
        
        # Draw the hit region
        start = hsp.query_start
        end = hsp.query_end
        ax.barh(y, end - start, left=start, height=0.6, color=color, 
                edgecolor="black", linewidth=0.5)
        
        # Label
        label = aln.hit_id.split('|')[-1] if '|' in aln.hit_id else aln.hit_id[:12]
        id_pct = 100 * hsp.identities / hsp.align_length
        ax.text(query_len + 2, y, f"{label} ({id_pct:.0f}%)", 
                va="center", fontsize=8)
    
    # Query bar
    ax.barh(-1.2, query_len, left=1, height=0.4, color="steelblue", 
            edgecolor="black", linewidth=1)
    ax.text(query_len / 2, -1.2, "Query", ha="center", va="center", 
            fontweight="bold", color="white", fontsize=9)
    
    ax.set_xlim(0, query_len * 1.3)
    ax.set_ylim(max_hits, -2)
    ax.set_xlabel("Query position")
    ax.set_yticks([])
    ax.set_title("BLAST Hit Coverage Map")
    
    # Legend
    legend_items = [
        mpatches.Patch(color="red", label="E < 1e-50"),
        mpatches.Patch(color="orange", label="E < 1e-10"),
        mpatches.Patch(color="green", label="E < 0.01"),
        mpatches.Patch(color="gray", label="E >= 0.01"),
    ]
    ax.legend(handles=legend_items, loc="lower right", fontsize=8)
    
    plt.tight_layout()
    plt.show()


plot_blast_coverage(mock_record)

---

## 11. Summary

| Concept | Key Takeaway |
|---|---|
| **Why BLAST** | Smith-Waterman is too slow for database searches; BLAST uses heuristics |
| **Seed-and-extend** | Find short exact matches, extend promising ones, do full alignment only on best candidates |
| **BLAST variants** | Match query/DB type: blastp, blastn, blastx, tblastn, tblastx |
| **E-value** | Expected random hits at this score; depends on database size |
| **Bit score** | Normalized, database-independent score |
| **PSI-BLAST** | Iterative PSSM-based search for remote homologs |
| **Low-complexity** | Always filter; poly-X regions cause false positives |
| **Database choice** | SwissProt for quality; nr for completeness |
| **Identity thresholds** | >30% protein identity = likely homologous; 20-30% = twilight zone |

---

## Exercises

### Exercise 1: Identify a Mystery Protein

Use BLAST (web or programmatic) to identify the following protein sequence. What organism is it from? What is its function? What protein family does it belong to?

In [ ]:
# Exercise 1: Identify this protein
mystery = (
    "MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKD"
    "EAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRAALINMVFQMGETGVAGFTNSLRML"
    "QQKRWDEAAVNLAKSRWYNQTPNRAKRVITTFRTGTWDAYKNL"
)

# YOUR CODE HERE
# 1. Go to https://blast.ncbi.nlm.nih.gov/ and run blastp against swissprot
#    OR use: record = run_blast(mystery, "blastp", "swissprot")
# 2. What is the top hit? What organism?
# 3. What is the E-value and percent identity of the top hit?
# 4. What protein family does it belong to?
# 5. Are there hits from multiple species? What does this tell you?


### Exercise 2: Compare BLAST Databases

Search the human hemoglobin alpha sequence against both SwissProt and nr. Compare the results in terms of number of hits, taxonomic diversity, and runtime.

In [ ]:
# Exercise 2: Database comparison
hba = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLL"
    "SHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
)

# YOUR CODE HERE
# 1. Search against swissprot: run_blast(hba, "blastp", "swissprot")
# 2. Search against nr: run_blast(hba, "blastp", "nr")
# 3. Compare: how many hits? What organisms appear?
# 4. Are the E-values the same or different? Why?
# 5. Which database is better for which purpose?


### Exercise 3: Word Size Sensitivity

Demonstrate how word size affects BLAST sensitivity. Generate two related sequences (90% and 60% identity) and show the number of seed matches at different word sizes.

In [ ]:
# Exercise 3: Word size and sensitivity
import random

def mutate_sequence(seq, identity_pct):
    """Create a sequence with approximately the given percent identity."""
    aa = "ACDEFGHIKLMNPQRSTVWY"
    result = []
    for c in seq:
        if random.random() < identity_pct / 100:
            result.append(c)
        else:
            result.append(random.choice(aa))
    return "".join(result)


def count_word_matches(seq1, seq2, word_size):
    """Count exact word matches between two sequences."""
    words1 = set(seq1[i:i+word_size] for i in range(len(seq1) - word_size + 1))
    words2 = set(seq2[i:i+word_size] for i in range(len(seq2) - word_size + 1))
    return len(words1 & words2)


# YOUR CODE HERE
# 1. Generate a random protein of 200 aa
# 2. Create variants at 90%, 70%, 50%, 30% identity
# 3. For each variant, count word matches at W=2, 3, 4, 5
# 4. Plot: word matches vs identity for each word size
# 5. At what identity level does W=3 start finding very few matches?


### Exercise 4: Parse and Analyze Real BLAST Output

Write a function that takes tabular BLAST output and produces a summary report including: number of unique species, taxonomic distribution, and identity histogram.

In [ ]:
# Exercise 4: Analyze BLAST output

# Sample tabular BLAST data (realistic output)
sample_blast_output = """query1	sp|P69905|HBA_HUMAN	100.0	142	0	0	1	142	1	142	0.0	289
query1	sp|P01942|HBA_MOUSE	84.5	142	22	0	1	142	1	142	1e-85	252
query1	sp|P01946|HBA_RAT	83.1	142	24	0	1	142	1	142	3e-83	247
query1	sp|P01948|HBA_PIG	84.5	142	22	0	1	142	1	142	2e-84	250
query1	sp|P01958|HBA_CHICK	69.0	142	44	0	1	142	1	142	3e-68	205
query1	sp|P02024|HBB_HUMAN	29.9	147	100	3	1	141	1	147	5e-25	98
query1	sp|P02185|MYG_PHYCA	22.6	155	115	5	1	141	1	154	2e-15	65
query1	sp|P02100|HBE_HUMAN	35.2	145	91	2	1	141	1	147	1e-30	112
query1	sp|P68871|HBB_MOUSE	30.6	147	99	3	1	141	1	147	3e-25	99
query1	sp|P69891|HBG1_HUMAN	33.8	145	93	2	1	141	1	147	8e-28	105"""

# YOUR CODE HERE
# 1. Parse the tabular output into a DataFrame
# 2. Count how many hits are alpha-globins vs beta-globins vs myoglobin
# 3. Create a histogram of percent identities
# 4. Create a scatter plot of identity vs E-value
# 5. Which hits are above the twilight zone threshold?


### Exercise 5: BLAST Parameter Sensitivity

Using the NCBI BLAST web interface, search the same sequence with three different E-value thresholds (10, 0.001, 1e-10) and compare the number and quality of hits.

In [ ]:
# Exercise 5: E-value threshold effects

# YOUR CODE HERE
# 1. Use the hemoglobin alpha sequence
# 2. Run BLAST three times with E-value thresholds: 10, 0.001, 1e-10
# 3. Record: number of hits, lowest identity hit found, most distant organism
# 4. Make a table comparing the three searches
# 5. What would you recommend for:
#    a) Finding only confident orthologs?
#    b) Discovering the full globin family?
#    c) A first exploratory search?


### Exercise 6: Building a BLAST Automation Pipeline

Write a function that takes a list of FASTA sequences, BLASTs each one, and returns a consolidated report.

In [ ]:
# Exercise 6: Batch BLAST pipeline

def batch_blast_report(sequences, names, program="blastp", database="swissprot",
                       evalue=0.001):
    """Run BLAST for multiple sequences and produce a consolidated report.
    
    Parameters
    ----------
    sequences : list of str
        Query sequences.
    names : list of str
        Names for each query.
    
    Returns
    -------
    pd.DataFrame
        Consolidated results with columns: query_name, top_hit, e_value, identity, etc.
    """
    # YOUR CODE HERE
    # 1. Loop over sequences
    # 2. BLAST each one (with rate limiting: time.sleep(10) between queries)
    # 3. Extract top 3 hits for each
    # 4. Return a combined DataFrame
    pass


# Test sequences (don't actually run unless you have time)
test_seqs = [
    ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH", "Hemoglobin_A"),
    ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST", "Hemoglobin_B"),
]

print("batch_blast_report() defined.")
print("Uncomment and run to test (requires internet, ~2-3 min per query):")
print("  results = batch_blast_report([s for s,n in test_seqs], [n for s,n in test_seqs])")

---

## Further Reading

- Altschul SF, Gish W, Miller W, Myers EW, Lipman DJ (1990). Basic local alignment search tool. *J Mol Biol* 215:403-410.
- Altschul SF, Madden TL, Schaffer AA, et al. (1997). Gapped BLAST and PSI-BLAST: a new generation of protein database search programs. *Nucleic Acids Res* 25:3389-3402.
- Karlin S, Altschul SF (1990). Methods for assessing the statistical significance of molecular sequence features. *PNAS* 87:2264-2268.
- NCBI BLAST Help: https://blast.ncbi.nlm.nih.gov/doc/blast-help/
- BLAST+ Command Line Manual: https://www.ncbi.nlm.nih.gov/books/NBK279690/
- BioPython BLAST Tutorial: https://biopython.org/docs/latest/Tutorial/chapter_blast.html

**Previous notebook**: Pairwise Sequence Alignment -- the algorithmic foundation that BLAST builds upon.

**Next notebook**: Multiple Sequence Alignment -- aligning three or more sequences simultaneously.

---[< Previous: Pairwise Sequence Alignment](../03_Pairwise_Sequence_Alignment/01_pairwise_sequence_alignment.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Multiple Sequence Alignment: From Theory to Pra... >](../05_Multiple_Sequence_Alignment/01_multiple_sequence_alignment.ipynb)---